In [ ]:
from datetime import datetime
import os
import numpy as np
import netCDF4 as nc
from netCDF4 import Dataset
import pandas as pd
from netCDF4 import num2date
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import time
import sys
import cmaps
from scipy.interpolate import griddata, RegularGridInterpolator
BASE_PATH = '/home/yfdong/data/work/SMP'
DATA_PATH = '/raid61/yfdong/data/work/SMP'
sys.path.append(os.path.join(BASE_PATH, "code", "Library"))
from MeteoVarPlot import (add_quiverkey,
                          draw_Bias_subplot,draw_2DVAR_subplot,)
from PreprocessVar import (
    preprocess_var, 
    experiments_types,
    load_geodata, 
)
def interpolate_to_cmorph_by_griddata(src_var, src_points, CMORPH_lat, CMORPH_lon):
    dst_lat, dst_lon = CMORPH_lat, CMORPH_lon
    dst_lon_grid, dst_lat_grid = np.meshgrid(dst_lon, dst_lat)
    dst_points = np.array([dst_lat_grid.flatten(), dst_lon_grid.flatten()]).T
    # 强制类型转换
    dst_points = dst_points.astype(np.float32)
    REGRIDmethod = "linear"
    REGRIDmethod = "nearest"
    if src_var.shape[0]== 1 or src_var.ndim==2:
        dst_var = np.zeros((int(len(dst_lat)), int(len(dst_lon))))
        DstVar = griddata(src_points, (src_var).flatten(), dst_points, method=REGRIDmethod)
        dst_var[:] = DstVar.reshape((dst_lat_grid.shape[0], dst_lon_grid.shape[1]))
    elif src_var.shape[0] > 1 or src_var.ndim>=2:
        dst_var = np.zeros((int(src_var.shape[0]), int(len(dst_lat)), int(len(dst_lon))))
        for DateIndex in range(len(DateList)):  
            DstVar =  griddata(src_points, (src_var[DateIndex,:]).flatten(), dst_points, method=REGRIDmethod)
            dst_var[DateIndex,:] = DstVar.reshape((dst_lat_grid.shape[0], dst_lon_grid.shape[1]))
    else:
        print('Error: src_var.shape[0] is not 1 or greater than 1')
    return dst_var   

START_YEAR = 2000
END_YEAR = 2024
Years = list(np.arange(START_YEAR, END_YEAR+1, 1))
# 去掉异常年份
abnormal_years = [2008, 2016, 2021]
Years = np.array([year for year in Years if year not in abnormal_years])
DomainRegion = [20., 41.5, 105, 125] # 计算纬向平均的经纬度范围 [20.6, 41.5, 105, 125]
CtrlName, ExpName = 'Ctrl', 'FSM'
TotalVars = ['CMORPH', CtrlName, ExpName]
PlotDomainRegion = [21,41.5,108,123] # 绘图范围

In [ ]:
# 读取数据
import xarray as xr
# 读取数据
CalibrationStatus = "def" 
Wrf_VAR_NAME = 'RAINNC'
CMORPH_VAR_NAME = 'cmorph'
# 初始化结果字典
results_src_grid = {}
# 遍历每个年份
for Year in range(START_YEAR, END_YEAR+1):
    # 为每个年份创建一个包含三个实验的字典
    results_src_grid[Year] = {
        CtrlName: {},
        'CMORPH': {}
    }
# 读取地形数据
Wrf_lat, Wrf_lon, LandMask, DEM = load_geodata(GeoPath =os.path.join(DATA_PATH, "Domain/geo_em.d01.nc"), IF_LandMask=True)
experiment = experiments_types[0]
# Events_config = JJA_configs[1]
for Year in range(START_YEAR, END_YEAR+1):
    Timedelta = 24 # Hours, 读取wrfout降水逐日数据
    LAYER, SCALE_FACTOR = 0, 1
    # 读取事件的开始和结束时间
    StartReadMonth, EndReadMonth = 5, 8
    ReadStartDate, ReadEndDate = datetime(Year, StartReadMonth, 6), datetime(Year, EndReadMonth, 31)
    DateList = pd.date_range(start=ReadStartDate, end=ReadEndDate, freq=f'{Timedelta}h')
    # 读取WRF-S模拟数据
    WRF_DATA_PATH = os.path.join(DATA_PATH, f'output/dst/TotalVar/{Year}')
    results_src_grid[Year][CtrlName],_ = preprocess_var(WRF_DATA_PATH, CalibrationStatus, CtrlName, 
                    Wrf_VAR_NAME, DateList, ReadStartDate, ReadEndDate, LAYER, 
                    SCALE_FACTOR, LandMask, IF_LANDMASK=False)
    results_src_grid[Year][ExpName],_ = preprocess_var(WRF_DATA_PATH, CalibrationStatus, ExpName, 
                    Wrf_VAR_NAME, DateList, ReadStartDate, ReadEndDate, LAYER, 
                    SCALE_FACTOR, LandMask, IF_LANDMASK=False)    
    cache_path = os.path.join('/raid61/yfdong/data/OBS/Prec/CMORPH/China', f'CMORPH_V1.0_ADJ_8km-Daily_CN_{Year}.nc')
    ds = xr.open_dataset(cache_path)
        # 同时切片时间和空间
    data_subset = ds.sel(
        time=slice(ReadStartDate, ReadEndDate),
        lat=slice(DomainRegion[0], DomainRegion[1]),   # 注意：xarray 的 slice 是 inclusive 的
        lon=slice(DomainRegion[2], DomainRegion[3])
    )
    results_src_grid[Year]['CMORPH'] = data_subset.cmorph
    CMORPH_lat = (data_subset.lat).values
    CMORPH_lon = (data_subset.lon).values
   
# 统一分辨率
'''
To assess the performance of precipitation simulations, 
both the CHM_PRE and WCTRL model outputs were spatially interpolated onto the CMORPH grid using the bilinear interpolation method
Observation data: CHM_PRE and CMORPH, while the WCTRL is model output.
'''
from scipy.interpolate import griddata, RegularGridInterpolator
def create_domain_mask(lat, lon, domain):
   domain_lat_min, domain_lat_max, domain_lon_min, domain_lon_max = domain
   mask = (lat >= domain_lat_min) & (lat <= domain_lat_max) & (lon >= domain_lon_min) & (lon <= domain_lon_max)
   masked_array = xr.DataArray(np.where(mask, 1, np.nan), dims=[ 'south_north', 'west_east'])
   return masked_array

def extract_SubsetRegion_mask(var ,domain_mask):
    row_indices, col_indices = np.where(domain_mask == 1)  # 提取 ==1 的区域
    row_start, row_end = row_indices.min(), row_indices.max()
    col_start, col_end = col_indices.min(), col_indices.max()
    # 步骤2：切片获取子区域数据（保持二维结构）
    sub_var = var[row_start:row_end+1, col_start:col_end+1]
    return sub_var

def interpolate_to_cmorph_by_RegularGridInterpolator(src_var, src_lat, src_lon, CMORPH_lat, CMORPH_lon):
    dst_lat, dst_lon = CMORPH_lat, CMORPH_lon
    dst_lon_grid, dst_lat_grid = np.meshgrid(dst_lon, dst_lat)
    dst_points = np.array([dst_lat_grid.flatten(), dst_lon_grid.flatten()]).T
    # 强制类型转换
    dst_points = dst_points.astype(np.float32)
    # REGRIDmethod = "linear"
    REGRIDmethod = "nearest"
    if src_var.shape[0]== 1 or src_var.ndim==2:
        dst_var = np.zeros((int(len(dst_lat)), int(len(dst_lon))))
        print('SRC',src_var.shape,src_lat.shape, src_lon.shape)
        print('dst',dst_var.shape, dst_lat.shape, dst_lon.shape)
        interpolator = RegularGridInterpolator((src_lat, src_lon), src_var, method=REGRIDmethod,bounds_error=False, fill_value=None)
        dst_var[:] = interpolator(dst_points).reshape((len(dst_lat), len(dst_lon)))
    elif src_var.shape[0] > 1 or src_var.ndim>=2:
        dst_var = np.zeros((int(src_var.shape[0]), int(len(dst_lat)), int(len(dst_lon))))
        for DateIndex in range(len(DateList)):  
            interpolator = RegularGridInterpolator((src_lat, src_lon), src_var[DateIndex,:], method=REGRIDmethod,bounds_error=False, fill_value=None)
            dst_var[DateIndex,:] = interpolator(dst_points).reshape((len(dst_lat), len(dst_lon)))
    else:
        print('Error: src_var.shape[0] is not 1 or greater than 1')
    return dst_var    


# 初始化结果字典
results_CMORPH_grid = {}
# 遍历每个年份
for Year in range(START_YEAR, END_YEAR+1):
    # 为每个年份创建一个包含三个实验的字典
    results_CMORPH_grid[Year] = {
        CtrlName: {},
        ExpName: {},
        'CMORPH': {}
    }
# --------------------------------------------------------------------  

# --------------------- 提取CMORPH子区域 --------------------
dst_lon_grid , dst_lat_grid= np.meshgrid(CMORPH_lon, CMORPH_lat)
DomainMask = create_domain_mask(dst_lat_grid, dst_lon_grid, DomainRegion)
DomainMask = np.expand_dims(DomainMask,0) #广播
print(dst_lat_grid.shape, dst_lon_grid.shape)
for yearIndex, Year in enumerate(Years):
    results_CMORPH_grid[Year]['CMORPH'] = results_src_grid[Year]['CMORPH'] *DomainMask

# -------------------- 统一分辨率 (将wrfout结果插值到CMORPH网格)----------------------------
src_points = np.array([Wrf_lat.flatten(), Wrf_lon.flatten()]).T
from netCDF4 import date2num
# 添加tqdm进度条
from tqdm import tqdm
WrfVars = [CtrlName, ExpName]
total_iterations = len(Years) * len(WrfVars)
WRF_SCALE_FACTOR = 24
with tqdm(total=total_iterations, desc="Processing CMORPH interpolation") as pbar:
    for yearIndex, Year in enumerate(Years):
        StartReadMonth, EndReadMonth = 5, 8
        ReadStartDate, ReadEndDate = datetime(Year, StartReadMonth, 6), datetime(Year, EndReadMonth, 31)
        DateList = pd.date_range(start=ReadStartDate, end=ReadEndDate, freq=f'{Timedelta}h')
        for varIndex, var in enumerate(WrfVars):
            cache_path = os.path.join(DATA_PATH, f'Evaluation/Obs/CMORPH/dst/{var}_{Year}_Daily.nc')
            # print(Year, var)
            if os.path.exists(cache_path):
                # print("File exists, reading from cache: ", cache_path)
                with Dataset(cache_path, 'r') as cache_nc:
                    CMORPH_lat = cache_nc.variables['lat'][:]
                    CMORPH_lon = cache_nc.variables['lon'][:]
                    time_var = cache_nc.variables['time']
                    # units = time_var.units      # 应为 'days since 1970-01-01 00:00:00'
                    # calendar = time_var.calendar  # 应为 'standard'
                    # DateList = num2date(time_var, units=units, calendar=calendar)
                    results_CMORPH_grid[Year][var] = cache_nc.variables[var][:] * WRF_SCALE_FACTOR # mm/h -> mm/day
            else:
                # print("File does not exist: ", cache_path)
                # 读取源数据
                src_var = results_src_grid[Year][var]
                interpolate_data= interpolate_to_cmorph_by_griddata(src_var, src_points, CMORPH_lat, CMORPH_lon) # 非均匀网格插值
                interpolate_data = interpolate_data 
                # 进行插值计算     
                interpolate_data[interpolate_data>1000]=np.nan
                interpolate_data[interpolate_data<0]=np.nan
                results_CMORPH_grid[Year][var] = interpolate_data *DomainMask
                with Dataset(cache_path, 'w') as cache_nc:  
                    cache_nc.createDimension('time', None)
                    cache_nc.createDimension('lat', len(CMORPH_lat))
                    cache_nc.createDimension('lon', len(CMORPH_lon))
                    time_var = cache_nc.createVariable('time', 'f4', ('time',))
                    lat_var = cache_nc.createVariable('lat', 'f4', ('lat',))
                    lon_var = cache_nc.createVariable('lon', 'f4', ('lon',))
                    time_var.units = 'days since 1970-01-01 00:00:00'
                    time_var.calendar = 'standard'
                    # 创建主数据变量
                    data_var = cache_nc.createVariable(var, 'f4', ('time', 'lat', 'lon'), zlib=True)
                    time_var[:] = date2num(DateList.to_pydatetime(), units=time_var.units, calendar=time_var.calendar)
                    # 写入数据
                    data_var.units = 'mm/day'
                    lat_var[:] = CMORPH_lat
                    lon_var[:] = CMORPH_lon
                    data_var[:] = results_CMORPH_grid[Year][var]
        pbar.update(1)  # 每完成一次内层循环就更新进度            
# 插值地形
dstDEM = interpolate_to_cmorph_by_griddata(DEM, src_points, CMORPH_lat, CMORPH_lon)
dstLandMask = interpolate_to_cmorph_by_griddata(LandMask, src_points, CMORPH_lat, CMORPH_lon)

# ------------------------------- 处理为气候态数据（时间平均，计算空间评分）-----------------------------
ClimateSpatialMeanResults = {}
ClimateTemproalMeanResults = {}
# 
ndays, ny, nx = results_CMORPH_grid[2000][CtrlName].shape
for var in TotalVars:
    TemplateClimateVar = np.zeros((len(Years), ndays, ny, nx))
    for YearIndex, Year in enumerate(Years):
        TemplateClimateVar[YearIndex,:,:] = results_CMORPH_grid[Year][var] 
    ClimateSpatialMeanResults[var] = np.nanmean(np.nanmean(TemplateClimateVar, axis=0), axis=0) # (ny, nx)
    ClimateTemproalMeanResults[var] = np.nanmean(TemplateClimateVar, axis=0) # (ndays, ny, nx)，日历日平均

# 计算纬向平均的经纬度范围
HamerDomainRegion = [21,41.5, 110, 120] # 计算纬向平均的经纬度范围
dst_lon_grid , dst_lat_grid= np.meshgrid(CMORPH_lon, CMORPH_lat)
HamerDomainMask = create_domain_mask(dst_lat_grid, dst_lon_grid, DomainRegion)
HamerDomainMask3D = np.expand_dims(HamerDomainMask,0) #广播
HamerResults = {}
for var in TotalVars:
    HamerResults[var] = ClimateTemproalMeanResults[var] *HamerDomainMask3D    


In [ ]:
# 可视化
import cartopy.feature as cfeature
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.ticker as ticker
import cmaps
CMAP = cmaps.WhiteBlueGreenYellowRed


# 定义公共参数
Plt_CommonParams = {
    'width': 3.2,
    'Height': 2.3,
    'dpi': 250,
    'title_labelsize': 7,
    'xy_labelsize': 6,
    'cbar_Legend_size': 6,
    'Grid': True,
    'Save': False,
    'Shp': False,
    'River': False,
    'Basin': False,
    'Topo': True,
    'shrink': 0.8,
    'fraction': 0.05,
    # 'Cbar': False
}
i = 0
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'grey'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.5     # 边框线宽
NumCols = len(TotalVars)
NumRows = 1
NumTicks = 5
pad = 0.1
UNIT = 'mm/day'
cbar = 'subcbar'
fig, axes = plt.subplots(nrows=NumRows, ncols=NumCols, figsize=(1.7*NumCols, 2.2*NumRows), dpi=350 ,sharex=True ,subplot_kw={'projection': ccrs.PlateCarree()})
# axs = axes.flatten()
TotalVars = ['CMORPH', CtrlName, ExpName]
plotMIN ,plotMAX1 = 0, 13
subsample = 1
# 绘制降水观测数据
Var = ClimateSpatialMeanResults['CMORPH']
lon, lat = CMORPH_lon, CMORPH_lat
ax=axes[0]
DomainRegion = [21.6,41.5,107.7,123.1]
CMAP_Diff = cmaps.ncl_default
draw_2DVAR_subplot(
    DomainRegion, 0 ,plotMAX1, lon, lat, 
    Var[:] , '(a) Obs (mm/day)', cbarTitle=f'[{UNIT}]', 
    CMAP=CMAP, save_name=False,
    **Plt_CommonParams, NumTicks = NumTicks, pad=pad, DEM = dstDEM, ax=ax, 
    Cbar=cbar, IF_Rectangle=False
    )
# 绘制Ctrl与观测的偏差
Var = ClimateSpatialMeanResults[CtrlName] - ClimateSpatialMeanResults['CMORPH']
lon, lat = CMORPH_lon, CMORPH_lat
ax=axes[1]
draw_Bias_subplot(
    DomainRegion, 4, lon, lat, 
    Var[:] , '(b) CTRL-Obs (mm/day)', cbarTitle=f'[{UNIT}]', 
    CMAP=CMAP_Diff, save_name=False,
    **Plt_CommonParams,  NumTicks = NumTicks, pad=pad, DEM = dstDEM, ax=ax, 
    Cbar=cbar, IF_Rectangle=False
    )  
# 绘制FSM与观测的偏差
Var = ClimateSpatialMeanResults['FSM'] - ClimateSpatialMeanResults['CMORPH']
lon, lat = CMORPH_lon, CMORPH_lat
ax=axes[2]
draw_Bias_subplot(
    DomainRegion, 4, lon, lat, 
    Var[:] , '(c) FSM-Obs (mm/day)', cbarTitle=f'[{UNIT}]', 
    CMAP=CMAP_Diff, save_name=False,
    **Plt_CommonParams,  NumTicks = NumTicks, pad=pad, DEM = dstDEM, ax=ax, 
    Cbar=cbar, IF_Rectangle=False
    ) 
for axIdx, ax in enumerate(axes):

    ax.add_feature(cfeature.COASTLINE, edgecolor='black', linewidth=0.5, zorder=2)  # 绘制黑色海岸线
    ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=2)  # 海洋填充为白色
    # ax.set_title(var, fontsize=Plt_CommonParams['xy_labelsize']+1 ) #,fontweight='bold'
    if axIdx != 0:
        ax.set_yticks([])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=5))
    # ax.tick_params(axis='x', which='both', bottom=True, labelbottom=True)    
# 调整子图间距
plt.subplots_adjust( wspace=0.05,hspace=0.05) #bottom=0.1, top=0.96,
plt.show()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import ticker
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Rectangle
import cmaps  # 需要确保安装了cmaps库

# ----------------- 修改1：重写 add_cbar 函数，使其基于传入的 ax 创建独立的 colorbar -----------------
def add_cbar(Cbar, ax, fig, contour, cbar_Legend_size, cbarTitle, numticks=5):
    if Cbar == True:
        # 为当前 ax 添加垂直方向的独立 colorbar
        cbar = fig.colorbar(contour, ax=ax, orientation='vertical', 
                            shrink=0.8, pad=0.02, aspect=15)
        cbar.locator = ticker.MaxNLocator(nbins=numticks)
        cbar.minorticks_off()
        cbar.locator = ticker.LinearLocator(numticks=numticks)
        cbar.ax.tick_params(which='both', width=0.5, length=1.5)
        cbar.ax.tick_params(labelsize=cbar_Legend_size)
        
        # 自定义格式化函数
        def format_tick(tick_value, pos):
            if isinstance(tick_value, float):

                return str(int(tick_value))

            else:
                return str(tick_value)
        
        cbar.formatter = ticker.FuncFormatter(format_tick)
        cbar.update_ticks()
        
        if cbarTitle:
            cbar.set_label(cbarTitle, fontsize=cbar_Legend_size)
            
    elif Cbar == "subcbar":
        cbar = fig.colorbar(contour, ax=ax, orientation='horizontal', 
                            shrink=1.0, fraction=0.05, pad=0.12, aspect=35)
        cbar.locator = ticker.LinearLocator(numticks=numticks)
        cbar.ax.tick_params(which='both', length=1.2)
        cbar.ax.tick_params(labelsize=cbar_Legend_size-1.5)
    pass

# ----------------- 修改2：函数签名增加 plotMIN 参数 -----------------
def draw_Prec_temporal_subplot(
                Domain, plotMIN, plotMAX, CMAP, 
                LAT, VAR, DateVAR, title, cbarTitle, save_name=True,
                Height=2.3, width=3.2,
                dpi=350,
                title_labelsize=9, xy_labelsize=8, cbar_Legend_size=8, date_interval="weekly",
                Grid=True, Save=False,
                ax=None, Cbar=False):

    # print(save_name)
    if ax is None:
        fig = plt.figure(figsize=(width, Height), dpi=dpi)
        ax = fig.subplots(1, 1, subplot_kw={'projection': ccrs.PlateCarree()})
    else:
        fig = ax.figure
        
    ax.set_facecolor('#F5F5F5')
    
    # ----------------- 修改3：使用传入的 plotMIN 和 plotMAX 构建 levels -----------------
    levels = np.linspace(plotMIN, plotMAX, 31)
    
    VAR_transposed = VAR.T
    norm = BoundaryNorm(levels, ncolors=CMAP.N)
    DateList_num = mdates.date2num(DateVAR)
    
    # ----------------- 修改4：根据是否包含负值自动判断 extend 模式 -----------------
    extend_mode = 'both' if plotMIN < 0 else 'max'
    
    contour = ax.contourf(DateList_num, LAT, VAR_transposed, cmap=CMAP, levels=levels, extend=extend_mode, norm=norm)

    if date_interval == 'daily':
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    elif date_interval == 'weekly':
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    elif date_interval == 'monthly':
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%m'))
    elif date_interval == 'yearly':
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    if Cbar == True:
        add_cbar(Cbar, ax, fig, contour, cbar_Legend_size, cbarTitle, numticks=numticks)

    ax.set_title(title, fontsize=title_labelsize, loc='left', x=-0.1)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d\n%b'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=14))
    
    ax.minorticks_on()
    ax.tick_params(axis="both", which="major", direction="in", width=0.9, length=5, color='#363636')
    ax.tick_params(axis="both", which="minor", direction="in", width=0.75, length=3, color='#363636')
    ax.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha=0.3)
    ax.grid(True, which="minor", linestyle=":", color="gray", linewidth=0.75, alpha=0.3)
    
    LATmin, Latmax, LONmin, LONmax = Domain
    ax.set_ylim(LATmin, Latmax)

    return contour, levels

# ================= 主绘图流程 =================

images = []
PltTotalVars = ['CMORPH', ExpName, 'FSM-CTRL']
NumRows = len(PltTotalVars)
NumCols = 1

fig, axes = plt.subplots(nrows=NumRows, ncols=NumCols, 
                         figsize=(5.*NumCols*0.65, 2.*NumRows*0.65), 
                         dpi=350, sharex=True, sharey=True)
fontszie = 8

for varIndex, var in enumerate(PltTotalVars):
    if var == 'CMORPH':
        Var = np.nanmean(HamerResults[var], axis=2)
        plotMIN, plotMAX1 = 0, 15  # 明确指定 plotMIN 为 0
        current_cmap = CMAP
        numticks = 6
        
    elif var == 'FSM-CTRL':
        Var = np.nanmean(HamerResults['FSM'], axis=2)*1 - np.nanmean(HamerResults['Ctrl'], axis=2)*1
        plotMIN, plotMAX1 = -2, 2  # 明确指定 plotMIN 为 -2
        current_cmap = cmaps.ncl_default  # 使用红蓝差异色标
        numticks = 5
    else:
        Var = np.nanmean(HamerResults[var], axis=2)*1
        plotMIN, plotMAX1 = 0, 15  # 明确指定 plotMIN 为 0
        current_cmap = CMAP
        numticks = 6

    lon, lat = CMORPH_lon, CMORPH_lat
    ax = axes[varIndex]
    
    # ----------------- 修改5：调用时传入 plotMIN -----------------
    contour, levels = draw_Prec_temporal_subplot(
                    DomainRegion, plotMIN, plotMAX1, current_cmap,  # 传入 plotMIN
                    lat, Var, DateList, title=None, cbarTitle=None, save_name=True,
                    Height=2.3, width=3.2,
                    dpi=350,
                    title_labelsize=9, xy_labelsize=7, cbar_Legend_size=7, date_interval="monthly",
                    Grid=True, Save=False,
                    ax=ax, Cbar=True) 
    
    ax.set_ylabel(var, fontsize=fontszie+3, labelpad=5)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))

plt.subplots_adjust(wspace=0.05, hspace=0.15)
plt.show()